#### This particular notebook details the CellCharter spatial neighborhood analysis performed on the combined Xenium Datasets 1 and 3 (restricted to their 275 shared genes).

#### Required input files:

* Annotated cell-based data object (we used the processed version of the combined Xenium Datasets 1 and 3 object)

Environment: Please create and activate the conda environment provided in cellcharter_env.yaml before running this notebook.

Additional note: The autok.fit() step is computationally intensive, so we recommend using a GPU node if available. Once the CellCharter cluster assignments have been added to the data object, you can switch to a CPU node for subsequent analyses.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq

import anndata

import os
import cellcharter as cc
import scvi

from pathlib import Path
import re

import pickle

from matplotlib.patches import Patch
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from lightning.pytorch import seed_everything

seed_everything(12345)
scvi.settings.seed = 12345

In [ ]:
# Load in dataset
XeniumData = sc.read_h5ad('/path/26_03_31_Xenium_CombinedDatasets1and3_275SharedGenes_Annotated.h5ad')

# View
XeniumData

## Compute CellCharter spatial clusters

Start by creating a spatial graph using Squidpy functions. 

Unlike with Squidpy, you shouldn't subset data to only the condition of interest before creating the spatial graph. CellCharter is more analogous to traditional clustering, where you want to capture the full information and variation present across the entire dataset.

In [ ]:
## Build the spatial neighbors graphs

# Using delaunay = True as my approach

sq.gr.spatial_neighbors(
    XeniumData,
    library_key='Core_Info', # Use the sample variable
    coord_type="generic",
    delaunay=True,
)

In [ ]:
## Plot data

plt.rcParams['font.size'] = 7

sq.pl.spatial_scatter(
    XeniumData,
    shape=None,
    title=['Full dataset'],
    color=[
        "HS", # Using HS instead of cores because cores has too many unique values
    ],
    connectivity_key='spatial_connectivities',
    size=1,
    figsize=[18,18],
)

Use CellCharter's remove_long_links function to remove long-distance neighbor connections

Note: These typically represent a small fraction of connections, and their inclusion or exclusion has not noticeably affected our results

In [ ]:
cc.gr.remove_long_links(XeniumData)

In [ ]:
## Plot data

plt.rcParams['font.size'] = 7

sq.pl.spatial_scatter(
    XeniumData,
    shape=None,
    title=['Full dataset'],
    color=[
        "HS", # Using HS instead of cores because cores has too many unique values
    ],
    connectivity_key='spatial_connectivities',
    size=1,
    figsize=[18,18],
)

In [ ]:
cc.gr.aggregate_neighbors(XeniumData, n_layers=3, use_rep='X_pca_harmony', out_key='X_cellcharter')

# Note: The XeniumCombinedDatasets1and3 data object is the only one where X_pca_harmony exists. For the other objects, we let the harmony embedding override X_pca.
# Layers is how many k-hops away we're looking

In [ ]:
# Following guidance from CellCharter readthedocs tutorial

autok = cc.tl.ClusterAutoK(
    n_clusters=(2,10), 
    max_runs=10,
    convergence_tol=0.001
)

In [ ]:
# We recommend using a GPU node for this step, if possible. Very slow otherwise

autok.fit(XeniumData, use_rep='X_cellcharter')

In [ ]:
# Plot stability graph

cc.pl.autok_stability(autok)

If you don't manually set a k, then it will automatically use the number of clusters associated with the highest stability (based on the plot).

We suggest saving multiple k values

In [ ]:
# Save for more than 1 k value (so that you can compare options and see what makes biological sense)
# Note: We also tried k = 2, 5, and 6, but wanted more fine spatial niches

for k in [8, 9, 10]:
    XeniumData.obs[f'cluster_cellcharter_k{k}'] = autok.predict(
        XeniumData, use_rep='X_cellcharter', k=k
    )

In [ ]:
## Save object with cellcharter clusters

# Set base_path
base_path = "/path"

# Save object
XeniumData.write_h5ad(
    f"{base_path}/XeniumDatasets1and3_WithCellCharterClusters.h5ad"
)

In [ ]:
## Save autok object and stability plot -- This way, you can re-plot your stability plot and add additional k values to your dataset in the future if desired

plot_path = os.path.join(base_path, "CellCharter_AutoK_StabilityPlot.png")
autok_path = os.path.join(base_path, "CellCharter_AutoK_Object.pkl")

# -----------------------------
# Save AutoK stability plot
# -----------------------------
cc.pl.autok_stability(autok)
plt.tight_layout()
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Save AutoK object
# -----------------------------
with open(autok_path, "wb") as f:
    pickle.dump(autok, f)

print(f"AutoK plot saved to: {plot_path}")
print(f"AutoK object saved to: {autok_path}")

In [ ]:
## Example code as to how you would load back in your autok object

# Path to your saved object
#autok_path = os.path.join(base_path, "CellCharter_AutoK_Object.pkl")

# Load it
#with open(autok_path, "rb") as f:
#    autok = pickle.load(f)

#print("AutoK object loaded successfully")

## Examine CellCharter results, read data object back in if necessary

GPU node isn't needed anymore

In [ ]:
## Settings

# We're just using 1 k value (8) for this notebook
ks = [8]

# Users can modify this to run for additional k values
# ks = [8, 9, 10]

condition_col = "Condition"
hs_col = "HS"

# Set base_path again if needed
#base_path = "/path"

In [ ]:
# Read in object with CellCharter clusters (if needed)
#XeniumData = sc.read_h5ad(
#    f"{base_path}/XeniumDatasets1and3_WithCellCharterClusters.h5ad"
#)

# View
XeniumData

In [ ]:
## Output and save pivot tables

tables = {}

for k in ks:
    col = f'cluster_cellcharter_k{k}'
    
    # Group + count
    count_table = (
        XeniumData.obs
        .groupby([col, 'Free_annotations_SeparatePreprocessing_MergedNames_260310'])
        .size()
        .reset_index(name='count')
    )
    
    # Pivot
    pivot_table = count_table.pivot_table(
        index=col,
        columns='Free_annotations_SeparatePreprocessing_MergedNames_260310',
        values='count',
        fill_value=0
    ).sort_index()
    
    tables[k] = pivot_table
    
    # Display
    print(f"\n===== Cluster–CellType Count Table (k = {k}) =====")
    display(pivot_table)
    
    # Save using naming scheme
  #  out_path = f"{base_path}/XeniumDatasets1and3_WithCellCharterClusters_k{k}_AnnotationCounts.csv"
  #  pivot_table.to_csv(out_path)

  #  print(f"Saved: {out_path}")

In [ ]:
for k in ks:
    group_col = f"cluster_cellcharter_k{k}"
    
    print(f"\n===== Running Enrichment for k = {k} =====")

    # Compute enrichment statistics
    cc.gr.enrichment(
        XeniumData,
        group_key=group_col,
        label_key='Free_annotations_SeparatePreprocessing_MergedNames_260310'
    )
    
    # Plot enrichment
    cc.pl.enrichment(
        XeniumData,
        group_key=group_col,
        label_key='Free_annotations_SeparatePreprocessing_MergedNames_260310',
        group_cluster=True,
        label_cluster=True,
        figsize=(7, 5),
        fontsize=8,
        dot_scale=3,
    )

    # Get current figure and save it
  #  fig = plt.gcf()
  #  out_path = f"{base_path}/XeniumDatasets1and3_WithCellCharterClusters_k{k}_EnrichmentPlot.png"
  #  fig.savefig(out_path, dpi=300, bbox_inches='tight')

  #  print(f"Saved PNG for k={k}: {out_path}")

    # Display the figure in the notebook
    plt.show()

In [ ]:
### Cell type enrichment heatmap

## Loop through k values

label_col = "Free_annotations_SeparatePreprocessing_MergedNames_260310"

for k in ks:
    print(f"\nRunning enrichment for k={k}...\n")
    
    group_col = f"cluster_cellcharter_k{k}"
    
    # Skip if column doesn't exist
    if group_col not in XeniumData.obs.columns:
        print(f"{group_col} not found — skipping")
        continue

    # -----------------------------
    # Compute enrichment
    # -----------------------------
    cc.gr.enrichment(
        XeniumData,
        group_key=group_col,
        label_key=label_col
    )

    # -----------------------------
    # Pull enrichment matrix
    # -----------------------------
    enrich_key = f"{group_col}_{label_col}_enrichment"
    enrich_obj = XeniumData.uns[enrich_key]
    enrich_mat = enrich_obj["enrichment"]

    if not isinstance(enrich_mat, pd.DataFrame):
        enrich_mat = pd.DataFrame(enrich_mat)

    enrich_mat = enrich_mat.T

    # -----------------------------
    # Reorder rows by dendrogram
    # -----------------------------
    dendro_key = f"dendrogram_{label_col}"
    dendro_order = XeniumData.uns[dendro_key]["categories_ordered"]

    row_order = [x for x in dendro_order if x in enrich_mat.index]
    enrich_mat = enrich_mat.loc[row_order]

    # -----------------------------
    # Handle inf values
    # -----------------------------
    finite_vals = enrich_mat.replace([np.inf, -np.inf], np.nan).values
    finite_min = np.nanmin(finite_vals)

    enrich_mat = enrich_mat.replace(-np.inf, finite_min)
    enrich_mat = enrich_mat.replace(np.inf, np.nan)

    # -----------------------------
    # Plot
    # -----------------------------
    plt.figure(figsize=(8, max(6, 0.28 * enrich_mat.shape[0])))

    ax = sns.heatmap(
        enrich_mat,
        cmap="Reds",
        vmin=0,
        vmax=2,
        linewidths=0.3,
        linecolor="white",
        cbar_kws={
            "label": "Cell type enrichment (log2FC)",
            "ticks": [0, 1, 2]
        }
    )

    ax.set_xlabel(f"CellCharter k={k} cluster", fontsize=12)
    ax.set_ylabel("Fine annotations", fontsize=12)

    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # -----------------------------
    # Save
    # -----------------------------
 #   out_path = f"{base_path}/XeniumDatasets1and3_CellCharter_k{k}_FineAnnotation_EnrichmentHeatmap.png"
  #  plt.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.show()
#    plt.close()

#    print(f"Saved: {out_path}")

In [ ]:
### Loop through k values

# ---------------------------------------------
# Settings
# ---------------------------------------------
allowed_conditions = ["HC", "PRE_VDZ_NR", "PRE_VDZ_R"]
condition_order = ["HC", "PRE_VDZ_R", "PRE_VDZ_NR"]

# ------------------------------------------------------------
# Loop through k values
# ------------------------------------------------------------
for k in ks:
    print(f"\n==================== k = {k} ====================\n")

    cluster_col = f"cluster_cellcharter_k{k}"

    # Skip if column doesn't exist
    if cluster_col not in XeniumData.obs.columns:
        print(f"{cluster_col} not found — skipping")
        continue

    df = XeniumData.obs[[hs_col, condition_col, cluster_col]].copy()

    # Keep only desired conditions
    df = df[df[condition_col].isin(allowed_conditions)].copy()
    df = df.dropna(subset=[hs_col, cluster_col, condition_col])

    # Count cells per condition and cluster
    count_df = (
        df.groupby([condition_col, cluster_col], observed=False)
          .size()
          .reset_index(name="count")
    )

    pivot = (
        count_df.pivot(index=condition_col, columns=cluster_col, values="count")
                .fillna(0)
    )

    # Reorder conditions
    condition_order_present = [c for c in condition_order if c in pivot.index]
    pivot = pivot.loc[condition_order_present]

    # Convert to proportions
    props = pivot.div(pivot.sum(axis=1), axis=0)

    # ------------------------------------------------------------
    # Sort clusters numerically
    # ------------------------------------------------------------
    def sort_key(x):
        try:
            return int(x)
        except Exception:
            return x

    clusters = sorted(props.columns, key=sort_key)

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------
    x = np.arange(len(condition_order_present))
    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

    bottom = np.zeros(len(condition_order_present))
    for cl in clusters:
        vals = props[cl].values
        ax.bar(x, vals, bottom=bottom, label=str(cl))
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(condition_order_present, rotation=0, fontsize=14)
    ax.set_ylabel("Cluster proportion", fontsize=14)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis="y", labelsize=12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        title=f"{cluster_col}",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=12,
        title_fontsize=14,
        frameon=False,
    )

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
  #  out_path = f"{base_path}/ClusterProportions_stacked_k{k}_Condition_Updated.png"
   # fig.savefig(out_path, dpi=300, bbox_inches="tight")

  #  print(f"Saved: {out_path}")

    plt.show()
    plt.close()

In [ ]:
## Stacked bar plots

# ---------------------------------------------
# Settings
# ---------------------------------------------
allowed_conditions = ["HC", "PRE_VDZ_NR", "PRE_VDZ_R"]
condition_order = ["HC", "PRE_VDZ_R", "PRE_VDZ_NR"]

# ---------------------------------------------
# Precompute HS-level metadata (condition per HS)
# ---------------------------------------------
hs_meta = (
    XeniumData.obs[[hs_col, condition_col]]
    .drop_duplicates()
    .groupby(hs_col, observed=False)
    .agg({condition_col: "first"})
)

for k in ks:
    cluster_col = f"cluster_cellcharter_k{k}"
    print(f"\n==================== k = {k} ====================\n")

    df = XeniumData.obs[[hs_col, condition_col, cluster_col]].copy()

    # Keep only desired conditions
    df = df[df[condition_col].isin(allowed_conditions)].copy()
    df = df.dropna(subset=[hs_col, cluster_col, condition_col])

    # Count cells per HS and cluster
    count_df = (
        df.groupby([hs_col, cluster_col], observed=False)
          .size()
          .reset_index(name="count")
    )

    pivot = (
        count_df.pivot(index=hs_col, columns=cluster_col, values="count")
                .fillna(0)
    )

    # Convert to proportions
    props = pivot.div(pivot.sum(axis=1), axis=0)

    # ------------------------------------------------------------
    # Order: condition first, then alphabetic HS within condition
    # ------------------------------------------------------------
    
    # Only keep HS actually present in props
    all_hs = props.index.tolist()
    
    # Restrict metadata to allowed conditions AND present HS
    meta_sub = hs_meta.loc[all_hs].copy()
    meta_sub = meta_sub[meta_sub[condition_col].isin(allowed_conditions)].copy()
    
    # Build ordered list: condition first, alphabetic within condition
    hs_order = []
    for cond in condition_order:
        hs_in_cond = meta_sub.index[meta_sub[condition_col] == cond].tolist()
        hs_order.extend(sorted(hs_in_cond))
    
    # Final safety restriction
    hs_order = [h for h in hs_order if h in props.index]
    
    # Reorder
    props = props.loc[hs_order]
    meta_sub = meta_sub.loc[hs_order]

    # ------------------------------------------------------------
    # Sort clusters numerically (if possible)
    # ------------------------------------------------------------
    def sort_key(x):
        try:
            return int(x)
        except Exception:
            return x

    clusters = sorted(props.columns, key=sort_key)

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------
    x = np.arange(len(hs_order))
    fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)

    bottom = np.zeros(len(hs_order))
    for cl in clusters:
        vals = props[cl].values
        ax.bar(x, vals, bottom=bottom, label=str(cl))
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(hs_order, rotation=90, fontsize=5)
    ax.set_ylabel("Cluster proportion", fontsize=12)
    ax.set_xlabel("HS", fontsize=12)
    ax.set_ylim(0, 1.18)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        title=f"{cluster_col}",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
        title_fontsize=10,
        frameon=False,
    )

    # ------------------------------------------------------------
    # Condition group bars
    # ------------------------------------------------------------
    y_bar_bottom = 1.03
    y_bar_top    = 1.08
    y_label      = 1.095

    def draw_group_bar(start_idx, end_idx, label_text):
        start_x = x[start_idx] - 0.45
        end_x   = x[end_idx] + 0.45
        width   = end_x - start_x

        ax.add_patch(plt.Rectangle(
            (start_x, y_bar_bottom),
            width,
            y_bar_top - y_bar_bottom,
            facecolor="black",
            edgecolor=None
        ))

        mid = (x[start_idx] + x[end_idx]) / 2
        ax.text(mid, y_label, label_text,
                ha="center", va="bottom", fontsize=12, color="black")

    start = 0
    for cond in condition_order:
        n_cond = int((meta_sub[condition_col] == cond).sum())
        if n_cond == 0:
            continue
        end = start + n_cond - 1
        draw_group_bar(start, end, cond)
        start = end + 1

    fig.subplots_adjust(bottom=0.35)

  #  out_path = f"{base_path}/ClusterProportions_stacked_k{k}_HS_ConditionOrdered.png"
  #  fig.savefig(out_path, dpi=300, bbox_inches="tight")
  #  print(f"Saved: {out_path}")

    plt.show()

In [ ]:
## Syntax format

# -------------------------
# Normalize condition labels across datasets
# -------------------------
condition_cleanup_map = {
    # IBD290 (already correct)
    "PRE_VDZ_R": "PRE_VDZ_R",
    "PRE_VDZ_NR": "PRE_VDZ_NR",

    # ICI480 → map to same format
    "UCPRE_R_VDZ": "PRE_VDZ_R",
    "UCPRE_NR_VDZ": "PRE_VDZ_NR",

    # (optional: keep consistent if present)
    "UCPOST_R_VDZ": "POST_VDZ_R",
    "UCPOST_NR_VDZ": "POST_VDZ_NR",
}

# Apply mapping
XeniumData.obs[condition_col] = (
    XeniumData.obs[condition_col]
    .astype(str)
    .str.strip()
    .map(condition_cleanup_map)
)

print("\nCondition counts AFTER renaming:")
print(XeniumData.obs[condition_col].value_counts(dropna=False))

In [ ]:
### Merge and rename CellCharter clusters

k = 8
orig_cluster_col = f"cluster_cellcharter_k{k}"
merged_cluster_col = f"{orig_cluster_col}_merged"

cluster_merge_map = {
    "0": "GALT-B-DC-S4_fibroblast_NR",
    "3": "Other",
    "4": "Other",
    "7": "Other",
    "2": "IEC_R",
    "6": "IEC_R",
    "1": "IAF-Monocyte-Neutrophil_NR",
    "5": "IAF-Monocyte-Neutrophil_NR",
}

XeniumData.obs[merged_cluster_col] = (
    XeniumData.obs[orig_cluster_col]
    .astype(str)
    .map(cluster_merge_map)
)

display(XeniumData.obs[[orig_cluster_col, merged_cluster_col]].drop_duplicates().sort_values(orig_cluster_col))
display(XeniumData.obs[merged_cluster_col].value_counts(dropna=False))

In [ ]:
### Output cell type proportion plot for merged CellCharter clusters

# ---------------------------------------------
# Plot UC_PRE_R vs UC_PRE_NR
# Exclude "Other" from plot and stats
# Only NR patients are colored
# Non-NR dots are smaller
# No connector lines
# Show uncorrected and corrected p-values (2 decimal places) instead of stars
# ---------------------------------------------

# -------------------------
# Inputs / settings
# -------------------------
cluster_col = merged_cluster_col

allowed_conditions = ["PRE_VDZ_R", "PRE_VDZ_NR"]
condition_order = ["PRE_VDZ_R", "PRE_VDZ_NR"]

condition_label_map = {
    "PRE_VDZ_R": "UC_PRE_R",
    "PRE_VDZ_NR": "UC_PRE_NR"
}

cluster_order = [
    "IEC_R",
    "IAF-Monocyte-Neutrophil_NR",
    "GALT-B-DC-S4_fibroblast_NR",
]

all_merged_clusters = [
    "IEC_R",
    "IAF-Monocyte-Neutrophil_NR",
    "GALT-B-DC-S4_fibroblast_NR",
    "Other",
]

x_label_map = {
    "IEC_R": "IEC_R",
    "IAF-Monocyte-Neutrophil_NR": "IAF-Monocyte-\nNeutrophil_NR",
    "GALT-B-DC-S4_fibroblast_NR": "GALT-B-DC-\nS4_fibroblast_NR",
}

# -------------------------
# Condition colors
# -------------------------
condition_colors = {
    "PRE_VDZ_R": "blue",
    "PRE_VDZ_NR": "red"
}

# -------------------------
# Only NR patient colors
# -------------------------
ucpre_nr_colors = [
    "maroon", "olive", "lightgreen", "paleturquoise", "darkcyan",
    "slategray", "darkslateblue", "orchid"
]

# -------------------------
# Natural sort for HS
# -------------------------
def natural_sort_key(s):
    s = str(s)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

# -------------------------
# Load / subset data
# -------------------------
df = XeniumData.obs[[hs_col, condition_col, cluster_col]].copy()
df = df[df[condition_col].isin(allowed_conditions)].copy()
df = df.dropna(subset=[hs_col, condition_col, cluster_col])

df[condition_col] = df[condition_col].astype(str)
df[hs_col] = df[hs_col].astype(str)
df[cluster_col] = df[cluster_col].astype(str)

df = df[df[cluster_col].isin(all_merged_clusters)].copy()

# -------------------------
# Build patient -> color map
# Only PRE_VDZ_NR gets unique colors
# Everything else is black
# -------------------------
patient_color_map = {}

for cond in condition_order:
    hs_vals = (
        df.loc[df[condition_col] == cond, hs_col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    hs_vals = sorted(hs_vals, key=natural_sort_key)

    if cond == "PRE_VDZ_NR":
        for i, hs in enumerate(hs_vals):
            patient_color_map[(cond, hs)] = ucpre_nr_colors[i % len(ucpre_nr_colors)]
    else:
        for hs in hs_vals:
            patient_color_map[(cond, hs)] = "black"

# -------------------------
# Count cells per HS and merged cluster
# Denominator includes ALL merged clusters, including Other
# -------------------------
count_df = (
    df.groupby([hs_col, condition_col, cluster_col], observed=False)
      .size()
      .reset_index(name="count")
)

pivot = count_df.pivot_table(
    index=[hs_col, condition_col],
    columns=cluster_col,
    values="count",
    fill_value=0,
    observed=False
)

pivot = pivot.reindex(columns=all_merged_clusters, fill_value=0)

props = pivot.div(pivot.sum(axis=1), axis=0).reset_index()

# -------------------------
# Wide -> long for plotting
# Only keep displayed/tested clusters here
# -------------------------
plot_df = props.melt(
    id_vars=[hs_col, condition_col],
    var_name="cluster",
    value_name="proportion"
)

plot_df[condition_col] = plot_df[condition_col].astype(str)
plot_df[hs_col] = plot_df[hs_col].astype(str)
plot_df["cluster"] = plot_df["cluster"].astype(str)

plot_df = plot_df[plot_df["cluster"].isin(cluster_order)].copy()

plot_df["cluster"] = pd.Categorical(
    plot_df["cluster"],
    categories=cluster_order,
    ordered=True
)

# -------------------------
# Statistics
# -------------------------
stats_rows = []

for cl in cluster_order:
    sub_df = plot_df[plot_df["cluster"] == cl]

    x = sub_df.loc[sub_df[condition_col] == "PRE_VDZ_R", "proportion"].astype(float).dropna()
    y = sub_df.loc[sub_df[condition_col] == "PRE_VDZ_NR", "proportion"].astype(float).dropna()

    if len(x) == 0 or len(y) == 0:
        stat, pval = np.nan, np.nan
    else:
        stat, pval = mannwhitneyu(x, y, alternative="two-sided")

    stats_rows.append({
        "cluster": cl,
        "group1": "PRE_VDZ_R",
        "group2": "PRE_VDZ_NR",
        "U": stat,
        "p_value": pval
    })

stats_df = pd.DataFrame(stats_rows)

mask = stats_df["p_value"].notna()
stats_df.loc[mask, "p_adj"] = multipletests(
    stats_df.loc[mask, "p_value"],
    method="fdr_bh"
)[1]
stats_df.loc[~mask, "p_adj"] = np.nan

def format_pval_text(p_raw, p_adj):
    if pd.isna(p_raw) or pd.isna(p_adj):
        return "p=NA\nFDR=NA"
    return f"p={p_raw:.2f}\nFDR={p_adj:.2f}"

stats_df["p_text"] = stats_df.apply(
    lambda row: format_pval_text(row["p_value"], row["p_adj"]),
    axis=1
)

print("\nStatistics table:")
print(stats_df[["cluster", "group1", "group2", "U", "p_value", "p_adj", "p_text"]])

# -------------------------
# X positions for dodged conditions
# -------------------------
n_hue = len(condition_order)
group_centers = np.arange(len(cluster_order), dtype=float)
offsets = np.linspace(-0.16, 0.16, n_hue)

x_lookup = {}
for i, cl in enumerate(cluster_order):
    for j, cond in enumerate(condition_order):
        x_lookup[(cl, cond)] = i + offsets[j]

# -------------------------
# Helper: significance brackets
# -------------------------
def add_pvalue_bracket(ax, x1, x2, y, text, h, lw=1.4):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=lw,
        c="black",
        clip_on=False
    )

    ax.text(
        (x1 + x2) / 2,
        y + h,
        text,
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="normal",
        linespacing=1.0
    )

# -------------------------
# Plot
# -------------------------
sns.set(style="ticks", rc={"figure.figsize": (10, 6)})

fig, ax = plt.subplots(figsize=(10, 6))

group_width = 0.55
box_width = group_width / len(condition_order)

# Draw boxes
for cl in cluster_order:
    for cond in condition_order:
        vals = plot_df.loc[
            (plot_df["cluster"] == cl) &
            (plot_df[condition_col] == cond),
            "proportion"
        ].dropna().values

        if len(vals) == 0:
            continue

        x_pos = x_lookup[(cl, cond)]

        bp = ax.boxplot(
            [vals],
            positions=[x_pos],
            widths=box_width * 0.9,
            patch_artist=True,
            showfliers=False,
            zorder=1
        )

        for patch in bp["boxes"]:
            color = sns.desaturate(condition_colors[cond], 0.75)
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
            patch.set_edgecolor("black")
            patch.set_linewidth(1.2)

        for median in bp["medians"]:
            median.set_color("black")
            median.set_linewidth(1.2)

        for whisker in bp["whiskers"]:
            whisker.set_color("black")
            whisker.set_linewidth(1.2)

        for cap in bp["caps"]:
            cap.set_color("black")
            cap.set_linewidth(1.2)

# -------------------------
# Patient points
# Only NR patients are colored
# Non-NR dots are smaller
# No connector lines
# -------------------------
rng = np.random.default_rng(42)

for cond in condition_order:
    cond_df = plot_df[plot_df[condition_col] == cond].copy()

    hs_vals = (
        cond_df[hs_col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    hs_vals = sorted(hs_vals, key=natural_sort_key)

    point_size = 28 if cond == "PRE_VDZ_NR" else 16

    for hs in hs_vals:
        hs_df = cond_df[cond_df[hs_col] == hs].copy()
        hs_df["cluster"] = pd.Categorical(
            hs_df["cluster"],
            categories=cluster_order,
            ordered=True
        )
        hs_df = hs_df.sort_values("cluster")

        for _, row in hs_df.iterrows():
            cl = str(row["cluster"])
            y = row["proportion"]

            base_x = x_lookup[(cl, cond)]
            jitter = rng.uniform(-0.04, 0.04)

            ax.scatter(
                base_x + jitter,
                y,
                s=point_size,
                alpha=1.0,
                color=patient_color_map[(cond, hs)],
                edgecolor="black",
                linewidth=0.25,
                zorder=3
            )

# -------------------------
# Legend
# -------------------------
legend_handles = [
    Patch(
        facecolor=sns.desaturate(condition_colors[cond], 0.75),
        edgecolor="black",
        linewidth=1.2,
        alpha=0.6
    )
    for cond in condition_order
]
legend_labels = [condition_label_map[cond] for cond in condition_order]

ax.legend(
    legend_handles,
    legend_labels,
    title="",
    frameon=False,
    loc="best",
)

# -------------------------
# Formatting
# -------------------------
ax.set_xticks(group_centers)
ax.set_xticklabels(
    [x_label_map[cl] for cl in cluster_order],
    rotation=0,
    ha="center",
    fontsize=11
)
ax.set_xlabel("")
ax.set_ylabel("Merged CellCharter k=8 cluster proportion\n(of total cells in HS)", fontsize=12)
ax.tick_params(axis="x", labelsize=11)
ax.tick_params(axis="y", labelsize=11)

sns.despine(offset=5, trim=False, ax=ax)

# -------------------------
# P-value brackets
# -------------------------
ymin = plot_df["proportion"].min()
ymax = plot_df["proportion"].max()
yrng = ymax - ymin if ymax > ymin else 1.0

base_pad = 0.10 * yrng
h = 0.02 * yrng

for _, row in stats_df.iterrows():
    cl = row["cluster"]
    p_text = row["p_text"]

    x1 = x_lookup[(cl, "PRE_VDZ_R")]
    x2 = x_lookup[(cl, "PRE_VDZ_NR")]

    cl_max = plot_df.loc[plot_df["cluster"] == cl, "proportion"].max()
    y = cl_max + base_pad

    add_pvalue_bracket(ax, x1, x2, y, p_text, h)

bottom_pad = 0.02 * yrng
ax.set_ylim(-bottom_pad, ymax + base_pad + 0.22 * yrng)

plt.tight_layout()

#out_path = f"{base_path}/ClusterProportions_boxplot_patients_k8merged_PREVDZR_vs_PREVDZNR_noOther_pvalueslisted.png"
#fig.savefig(out_path, dpi=300, bbox_inches="tight")
#print(f"Saved: {out_path}")

plt.show()

In [ ]:
## Function for stacked bar plots of merged CellCharter clusters

# ---------------------------------------------
# Settings
# ---------------------------------------------
cluster_order = [
    "IEC_R",
    "IAF-Monocyte-Neutrophil_NR",
    "GALT-B-DC-S4_fibroblast_NR",
    "Other",
]

cluster_colors = {
    "IEC_R": "blue",
    "IAF-Monocyte-Neutrophil_NR": "orange",
    "GALT-B-DC-S4_fibroblast_NR": "red",
    "Other": "grey",
}

# ---------------------------------------------
# Precompute HS-level metadata
# ---------------------------------------------
hs_meta = (
    XeniumData.obs[[hs_col, condition_col]]
    .drop_duplicates()
    .groupby(hs_col, observed=False)
    .agg({condition_col: "first"})
)

# ---------------------------------------------
# Helper function
# ---------------------------------------------
def make_stacked_bar_plot(
    adata,
    cluster_col,
    allowed_conditions,
    condition_order,
    out_path=None,
    legend_title=None,
    figsize=(14, 5),
):
    df = adata.obs[[hs_col, condition_col, cluster_col]].copy()
    df = df[df[condition_col].isin(allowed_conditions)].dropna(subset=[hs_col, cluster_col, condition_col])
    df = df[df[cluster_col].isin(cluster_order)].copy()

    count_df = (
        df.groupby([hs_col, cluster_col], observed=False)
        .size()
        .reset_index(name="count")
    )

    pivot = (
        count_df.pivot(index=hs_col, columns=cluster_col, values="count")
        .fillna(0)
        .reindex(columns=cluster_order, fill_value=0)
    )

    props = pivot.div(pivot.sum(axis=1), axis=0)

    all_hs = props.index.tolist()
    meta_sub = hs_meta.loc[all_hs].copy()
    meta_sub = meta_sub[meta_sub[condition_col].isin(allowed_conditions)].copy()

    hs_order = []
    for cond in condition_order:
        hs_in_cond = meta_sub.index[meta_sub[condition_col] == cond].tolist()
        hs_order.extend(sorted(hs_in_cond))

    hs_order = [h for h in hs_order if h in props.index]

    props = props.loc[hs_order]
    meta_sub = meta_sub.loc[hs_order]

    x = np.arange(len(hs_order))
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

    bottom = np.zeros(len(hs_order))
    for cl in cluster_order:
        vals = props[cl].values
        ax.bar(
            x,
            vals,
            bottom=bottom,
            label=str(cl),
            color=cluster_colors.get(cl, "black"),
            edgecolor="none",
            linewidth=0,
        )
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(hs_order, rotation=90, fontsize=9)
    ax.set_ylabel("Cluster proportion", fontsize=12)
    ax.set_xlabel("HS", fontsize=12)
    ax.set_ylim(0, 1.18)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        title=legend_title if legend_title is not None else cluster_col,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=12,
        title_fontsize=13,
        frameon=False,
    )

    y_bar_bottom = 1.03
    y_bar_top = 1.08
    y_label = 1.095

    def draw_group_bar(start_idx, end_idx, label_text):
        start_x = x[start_idx] - 0.45
        end_x = x[end_idx] + 0.45

        ax.add_patch(plt.Rectangle(
            (start_x, y_bar_bottom),
            end_x - start_x,
            y_bar_top - y_bar_bottom,
            facecolor="black",
            edgecolor=None,
        ))

        ax.text(
            (x[start_idx] + x[end_idx]) / 2,
            y_label,
            label_text,
            ha="center",
            va="bottom",
            fontsize=12,
            color="black",
        )

    start = 0
    for cond in condition_order:
        n_cond = int((meta_sub[condition_col] == cond).sum())
        if n_cond == 0:
            continue
        end = start + n_cond - 1
        draw_group_bar(start, end, cond)
        start = end + 1

    fig.subplots_adjust(bottom=0.35)

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {out_path}")

    plt.show()
    plt.close()

In [ ]:
# ---------------------------------------------
# Run for merged k8 only: PRE_VDZ_R + PRE_VDZ_NR (exclude HC)
# ---------------------------------------------

allowed_conditions_rr = ["PRE_VDZ_R", "PRE_VDZ_NR"]
condition_order_rr = ["PRE_VDZ_R", "PRE_VDZ_NR"]

cluster_col = "cluster_cellcharter_k8_merged"
#out_path = f"{base_path}/ClusterProportions_stacked_k8_merged_HS_PREVDZR_and_PREVDZNRonly.png"

make_stacked_bar_plot(
    adata=XeniumData,
    cluster_col=cluster_col,
    allowed_conditions=allowed_conditions_rr,
    condition_order=condition_order_rr,
   # out_path=out_path,
    legend_title=cluster_col,
    figsize=(10, 5),
)

In [ ]:
#### Merged k = 8 enrichment heatmap

label_col = "Free_annotations_SeparatePreprocessing_MergedNames_260310"
k = 8
group_col = f"cluster_cellcharter_k{k}_merged"

print(f"\nRunning enrichment for merged k={k}...\n")

# -----------------------------
# Compute enrichment
# -----------------------------
cc.gr.enrichment(
    XeniumData,
    group_key=group_col,
    label_key=label_col
)

# -----------------------------
# Pull enrichment matrix
# -----------------------------
enrich_key = f"{group_col}_{label_col}_enrichment"
enrich_mat = XeniumData.uns[enrich_key]["enrichment"]

if not isinstance(enrich_mat, pd.DataFrame):
    enrich_mat = pd.DataFrame(enrich_mat)

enrich_mat = enrich_mat.T

# -----------------------------
# Reorder rows by dendrogram
# -----------------------------
dendro_key = f"dendrogram_{label_col}"
dendro_order = XeniumData.uns[dendro_key]["categories_ordered"]

enrich_mat = enrich_mat.loc[[x for x in dendro_order if x in enrich_mat.index]]

# -----------------------------
# Reorder columns
# -----------------------------
merged_cluster_order = [
    "IEC_R",
    "IAF-Monocyte-Neutrophil_NR",
    "GALT-B-DC-S4_fibroblast_NR",
    "Other",
]

enrich_mat = enrich_mat[[x for x in merged_cluster_order if x in enrich_mat.columns]]

# -----------------------------
# Handle inf values
# -----------------------------
finite_vals = enrich_mat.replace([np.inf, -np.inf], np.nan).values
finite_min = np.nanmin(finite_vals)

enrich_mat = enrich_mat.replace(-np.inf, finite_min).replace(np.inf, np.nan)

# -----------------------------
# Plot
# -----------------------------
x_label_map = {
    "IEC_R": "IEC_R",
    "IAF-Monocyte-Neutrophil_NR": "IAF-Monocyte-\nNeutrophil_NR",
    "GALT-B-DC-S4_fibroblast_NR": "GALT-B-DC-\nS4_fibroblast_NR",
    "Other": "Other",
}

plt.figure(figsize=(10, max(6, 0.28 * enrich_mat.shape[0])))

ax = sns.heatmap(
    enrich_mat,
    cmap="Reds",
    vmin=0,
    vmax=2,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={
        "label": "Cell type enrichment (log2FC)",
        "ticks": [0, 1, 2]
    }
)

ax.set_xlabel(f"Merged CellCharter k={k} cluster", fontsize=12)
ax.set_ylabel("Fine annotations", fontsize=12)

ax.set_xticklabels(
    [x_label_map.get(col, col) for col in enrich_mat.columns],
    rotation=0,
    ha="center"
)

plt.yticks(rotation=0)
plt.tight_layout()

# -----------------------------
# Save
# -----------------------------
# out_path = f"{base_path}/XeniumDatasets1and3_MergedCellCharter_k{k}_FineAnnotation_EnrichmentHeatmap.png"
# plt.savefig(out_path, dpi=300, bbox_inches="tight")

plt.show()
plt.close()

# print(f"Saved: {out_path}")

In [ ]:
### Spatial scatter plots

# Smaller points + NO legend on plots
# Save separate legend PNG

# ---------------------------------------------
# Settings
# ---------------------------------------------
cluster_col = merged_cluster_col

cores_of_interest = [
    "22_21438_HS63_A1_PRE_VDZ_NR_S",
    "16_22085_HS64_A1_PRE_VDZ_NR_REC",
    "17_22087_HS59_A1_PRE_VDZ_R_RS",
    "HS34_VDZ_C_PRE_1_Mayo2_responder",
    "HS50_VDZ_RS_PRE_1_Mayo2_non-responder",
    "HS46_VDZ_LC_PRE_1_Mayo3_non-responder",
]

allowed_conditions = ["HC", "PRE_VDZ_NR", "PRE_VDZ_R"]

out_dir = os.path.join(base_path, "SpatialScatterPlots_K8CCMergedClusters_CoresOfInterest")
os.makedirs(out_dir, exist_ok=True)

fig_width = 5
fig_height = 5
point_size = 7

cluster_order = [
    "IEC_R",
    "IAF-Monocyte-Neutrophil_NR",
    "GALT-B-DC-S4_fibroblast_NR",
    "Other",
]

cluster_colors = {
    "IEC_R": "blue",
    "IAF-Monocyte-Neutrophil_NR": "orange",
    "GALT-B-DC-S4_fibroblast_NR": "red",
    "Other": "grey",
}

# ---------------------------------------------
# Helper
# ---------------------------------------------
def make_safe_filename(s):
    s = str(s)
    s = s.replace(" ", "_").replace("/", "_")
    s = re.sub(r"[^A-Za-z0-9_.-]", "_", s)
    return s

# ---------------------------------------------
# Filter data
# ---------------------------------------------
adata_sub = XeniumData[
    XeniumData.obs[condition_col].isin(allowed_conditions) &
    XeniumData.obs[core_col].astype(str).isin(cores_of_interest)
].copy()

# ---------------------------------------------
# CREATE LEGEND ONLY ONCE
# ---------------------------------------------
legend_handles = [
    Patch(color=cluster_colors[cl], label=cl)
    for cl in cluster_order
]

fig_leg, ax_leg = plt.subplots(figsize=(3, 3))
ax_leg.axis("off")

ax_leg.legend(
    handles=legend_handles,
    title="K8 CC merged clusters",
    loc="center",
    frameon=False,
    fontsize=10,
    title_fontsize=11,
)

legend_path = os.path.join(
    out_dir,
    "SpatialScatter_K8CCMergedClusters_LEGEND.png"
)
fig_leg.savefig(legend_path, dpi=300, bbox_inches="tight")
plt.close(fig_leg)

print(f"Saved legend: {legend_path}")

# ---------------------------------------------
# Plot per core (NO LEGEND)
# ---------------------------------------------
for core_id in cores_of_interest:

    adata_core = adata_sub[
        adata_sub.obs[core_col].astype(str) == str(core_id)
    ].copy()

    if adata_core.n_obs == 0:
        print(f"Skipping {core_id}")
        continue

    coords = adata_core.obsm["spatial"]

    adata_core.obs[cluster_col] = pd.Categorical(
        adata_core.obs[cluster_col].astype(str),
        categories=cluster_order,
        ordered=True
    )

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=300)

    for cl in cluster_order:
        mask = adata_core.obs[cluster_col] == cl
        if mask.sum() == 0:
            continue

        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=point_size,
            c=cluster_colors[cl],
            edgecolor="none",
            alpha=0.9
        )

    ax.set_title(f"Core: {core_id}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal", adjustable="box")
    # ax.invert_yaxis()

    plt.tight_layout()

    safe_core = make_safe_filename(core_id)
    out_path = os.path.join(
        out_dir,
        f"SpatialScatter_{safe_core}_K8CCMergedClusters_LargerPoints.png"
    )

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out_path}")

## Optional validation check: Comparing high gene signature expressing cells to CellCharter k = 8 neighborhood assignment

Must generate Xenium_GeneSignatureExpression.csv by running the SpatialScatterPlots notebook

In [ ]:
## Read in mean expression of gene signatures within individual cells from Datasets 1 and/or 3

# Important: File needs to be outputted using SpatialScatterPlots notebook
# It can be generated at the beginning of part 3; the output code is already written but commented out

file_path = f"{base_path}/Xenium_GeneSignatureExpression.csv"

Datasets1and3_GeneSignatureExpression_df = pd.read_csv(file_path)

display(Datasets1and3_GeneSignatureExpression_df)

In [ ]:
## Merge in CellCharter info (k = 8 only)

cluster_col = "cluster_cellcharter_k8"

Datasets1and3_GeneSignatureExpression_df = Datasets1and3_GeneSignatureExpression_df.merge(
    XeniumData.obs[[cluster_col]],
    left_on="cell_id",
    right_index=True,
    how="left"
)

## Sanity checks

display(Datasets1and3_GeneSignatureExpression_df)
print(f"{cluster_col} missing values:", Datasets1and3_GeneSignatureExpression_df[cluster_col].isna().sum())

In [ ]:
## Compute cells with top 10% and top 25% gene signature expression per dataset

Datasets1and3_GeneSignatureExpression_df["dataset_prefix"] = np.where(
    Datasets1and3_GeneSignatureExpression_df["cell_id"].str.startswith("ICI-480"),
    "ICI-480",
    np.where(
        Datasets1and3_GeneSignatureExpression_df["cell_id"].str.startswith("IBD-290"),
        "IBD-290",
        np.nan
    )
)

score_cols = ["MeanExp_R_IEC", "MeanExp_NR_IAF", "MeanExp_NR_GALT"]
quantile_map = {"Top10pct": 0.90, "Top25pct": 0.75}
dataset_prefixes = ["ICI-480", "IBD-290"]

for score_col in score_cols:
    for suffix, q in quantile_map.items():
        out_col = f"{score_col}_{suffix}"
        Datasets1and3_GeneSignatureExpression_df[out_col] = "No"

        for prefix in dataset_prefixes:
            mask = Datasets1and3_GeneSignatureExpression_df["dataset_prefix"] == prefix
            threshold = Datasets1and3_GeneSignatureExpression_df.loc[mask, score_col].quantile(q)

            Datasets1and3_GeneSignatureExpression_df.loc[
                mask & (Datasets1and3_GeneSignatureExpression_df[score_col] >= threshold),
                out_col
            ] = "Yes"

for suffix in quantile_map:
    for score_col in score_cols:
        col = f"{score_col}_{suffix}"
        print(f"\n{col}")
        print(pd.crosstab(
            Datasets1and3_GeneSignatureExpression_df["dataset_prefix"],
            Datasets1and3_GeneSignatureExpression_df[col]
        ))

for prefix in dataset_prefixes:
    mask = Datasets1and3_GeneSignatureExpression_df["dataset_prefix"] == prefix
    print(f"\n{prefix}")
    for score_col in score_cols:
        print(f"{score_col} Top10 threshold:", Datasets1and3_GeneSignatureExpression_df.loc[mask, score_col].quantile(0.90))
        print(f"{score_col} Top25 threshold:", Datasets1and3_GeneSignatureExpression_df.loc[mask, score_col].quantile(0.75))

In [ ]:
display(Datasets1and3_GeneSignatureExpression_df)

In [ ]:
# ---------------------------------------------
# See which CellCharter k=8 clusters top-signature cells are assigned to
# normalize="index" gives:
#   value = (# top-signature cells in cluster) / (total cells in cluster)
# ---------------------------------------------

df = Datasets1and3_GeneSignatureExpression_df.copy()

signature_cols = {
    "R_IEC": {"top10": "MeanExp_R_IEC_Top10pct", "top25": "MeanExp_R_IEC_Top25pct"},
    "NR_IAF": {"top10": "MeanExp_NR_IAF_Top10pct", "top25": "MeanExp_NR_IAF_Top25pct"},
    "NR_GALT": {"top10": "MeanExp_NR_GALT_Top10pct", "top25": "MeanExp_NR_GALT_Top25pct"},
}

def get_yes_proportion(data, cluster_col, signature_col):
    ct = pd.crosstab(data[cluster_col], data[signature_col], normalize="index")
    return ct["Yes"] if "Yes" in ct.columns else pd.Series(0, index=ct.index)

def sort_key(x):
    try:
        return int(x)
    except Exception:
        return x

df_k = df.dropna(subset=[cluster_col]).copy()

top10_df = pd.DataFrame({
    sig_name: get_yes_proportion(df_k, cluster_col, sig_info["top10"])
    for sig_name, sig_info in signature_cols.items()
}).fillna(0)

top25_df = pd.DataFrame({
    sig_name: get_yes_proportion(df_k, cluster_col, sig_info["top25"])
    for sig_name, sig_info in signature_cols.items()
}).fillna(0)

top10_df = top10_df.loc[sorted(top10_df.index, key=sort_key)]
top25_df = top25_df.loc[sorted(top25_df.index, key=sort_key)]

print(f"\n{'='*24} k = 8 {'='*24}")

print("\nPercentage of cells in each CellCharter cluster that fall within the top 10% of mean expression for each gene signature")
display((top10_df * 100).round(2))

print("\nPercentage of cells in each CellCharter cluster that fall within the top 25% of mean expression for each gene signature")
display((top25_df * 100).round(2))